In [3]:
import numpy as np

# ==========================================
# TRAFFIC SIGNAL OPTIMIZATION USING PSO
# ==========================================

# Traffic arrival rates (vehicles per second)
arrival_rate = np.array([0.40, 0.30, 0.50, 0.35])

# Vehicle discharge rates (vehicles per second)
service_rate = np.array([0.60, 0.60, 0.65, 0.60])

directions = ["North", "South", "East", "West"]

# Traffic signal parameters
cycle_time = 140
yellow_time = 3
all_red_time = 1

# Total green time available
total_green = cycle_time - 4 * (yellow_time + all_red_time)

# PSO parameters
particles = 40
iterations = 150

w = 0.7
c1 = 1.5
c2 = 1.5

min_green = 10
max_green = 60

# Simulation parameters
simulation_cycles = 10


# ==========================================
# QUEUE CALCULATION
# ==========================================

def calculate_queue(green_times):

    queues = np.zeros(4)
    max_queues = np.zeros(4)

    for cycle in range(simulation_cycles):

        for i in range(4):

            # Red time for each direction
            red_time = cycle_time - green_times[i]

            # Vehicles arriving during red signal
            arrivals = arrival_rate[i] * red_time

            # Vehicles arriving during green signal
            green_arrivals = (
                arrival_rate[i] * green_times[i]
            )

            # Maximum queue before green service
            queue_before_green = queues[i] + arrivals

            max_queues[i] = max(
                max_queues[i],
                queue_before_green
            )

            # Vehicles discharged during green
            departures = (
                service_rate[i] * green_times[i]
            )

            # Update queue
            queues[i] = max(
                0,
                queue_before_green
                + green_arrivals
                - departures
            )

            max_queues[i] = max(
                max_queues[i],
                queues[i]
            )

    return queues, max_queues


# ==========================================
# FITNESS FUNCTION
# ==========================================

def fitness(green_times):

    # Constraint penalty
    penalty = 0

    if abs(np.sum(green_times) - total_green) > 0.01:
        penalty += 100000

    if np.any(green_times < min_green):
        penalty += 100000

    if np.any(green_times > max_green):
        penalty += 100000

    # Calculate maximum queues
    _, max_queues = calculate_queue(green_times)

    # Minimize maximum queue
    objective = np.max(max_queues)

    return objective + penalty


# ==========================================
# INITIALIZE PARTICLES
# ==========================================

positions = np.random.dirichlet(
    np.ones(4), particles
) * total_green

velocities = np.random.uniform(
    -2, 2, (particles, 4)
)

pbest = positions.copy()

pbest_scores = np.array([
    fitness(p) for p in positions
])

best_index = np.argmin(pbest_scores)
gbest = pbest[best_index].copy()


# ==========================================
# PSO OPTIMIZATION
# ==========================================

for iteration in range(iterations):

    for i in range(particles):

        r1 = np.random.rand(4)
        r2 = np.random.rand(4)

        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest[i] - positions[i])
            + c2 * r2 * (gbest - positions[i])
        )

        positions[i] += velocities[i]

        # Apply green time limits
        positions[i] = np.clip(
            positions[i],
            min_green,
            max_green
        )

        # Normalize to total green time
        positions[i] = (
            positions[i]
            * total_green
            / np.sum(positions[i])
        )

        # Evaluate fitness
        score = fitness(positions[i])

        if score < pbest_scores[i]:
            pbest[i] = positions[i].copy()
            pbest_scores[i] = score

    best_index = np.argmin(pbest_scores)
    gbest = pbest[best_index].copy()


# ==========================================
# FINAL RESULTS
# ==========================================

final_queues, max_queues = calculate_queue(gbest)

print("\n======================================")
print(" OPTIMIZED TRAFFIC SIGNAL ALLOCATION")
print("======================================")

print("\nDirection | Green Time (seconds) | Maximum Queue (vehicles)")
print("-----------------------------------------------------------")

for i in range(4):

    print(
        f"{directions[i]:8s} | "
        f"{gbest[i]:8.2f} seconds | "
        f"({max_queues[i]:.0f} vehicles)"
    )

print("-----------------------------------------------------------")

print("\nTotal Green Time:",
      round(np.sum(gbest), 2), "seconds")

print("Maximum Queue at Intersection:",
      round(np.max(max_queues), 2), "vehicles")


 OPTIMIZED TRAFFIC SIGNAL ALLOCATION

Direction | Green Time (seconds) | Maximum Queue (vehicles)
-----------------------------------------------------------
North    |    35.20 seconds | (356 vehicles)
South    |    11.26 seconds | (356 vehicles)
East     |    54.20 seconds | (356 vehicles)
West     |    23.34 seconds | (356 vehicles)
-----------------------------------------------------------

Total Green Time: 124.0 seconds
Maximum Queue at Intersection: 355.82 vehicles
